In [79]:
import pandas as pd
import numpy as np

import torch as tc
import torch.nn as nn

import re
import random

from sklearn.model_selection import train_test_split
from collections import defaultdict
from torch.utils.data import TensorDataset, DataLoader

from tqdm import tqdm

device = tc.device("cuda" if tc.cuda.is_available() else "cpu")

In [80]:
df=pd.read_csv("poetry.csv") 
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [81]:
def clean(text):
    # text = text.lower()
    text = re.sub(r'["\'()\[\]{}]', '', text)
    text = re.sub(r'[.,!?;:]', '', text)
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [82]:
train_df, temp_df = train_test_split(df,test_size=0.2,random_state=42)
val_df, test_df = train_test_split(temp_df,test_size=0.50,random_state=42)

In [99]:
train_poems = [clean(p) for p in train_df["content"]]
val_poems   = [clean(p) for p in val_df["content"]]
test_poems  = [clean(p) for p in test_df["content"]]


train_text = "\n\n".join(train_poems)
train_tokens = train_text.split()
special_tokens = ["<PAD>", "<UNK>"]

vocab = special_tokens+sorted(set(train_tokens))

stoi = {word: idx for idx, word in enumerate(vocab)}
itos = {idx: word for idx, word in enumerate(vocab)}
vocab_size = len(vocab)

In [100]:
def encode(text):
    tokens = text.split()
    return [stoi.get(token, stoi["<UNK>"]) for token in tokens]

def decode(ids):
    return " ".join(itos[idx] for idx in ids)

In [101]:
train_data = encode(" ".join(train_poems))
val_data = encode(" ".join(val_poems))
test_data = encode(" ".join(test_poems))

In [102]:
def val_loss_bigram(data,prob_fn):
    total_loss = 0.0
    total_tokens = 0
    for i in range(len(data) - 1):
        context = data[i]
        target = data[i + 1]
        prob = prob_fn(context, target)
        prob = max(prob, 1e-10)
        total_loss += -np.log(prob)
        total_tokens += 1

    return total_loss / total_tokens

In [103]:
def tok_acc_bigram(data, pred_fn):
    correct = 0
    total = 0
    for i in range(len(data) - 1):
        context = data[i]
        target = data[i + 1]
        prediction = pred_fn(context)
        if prediction == target:
            correct += 1
        total += 1
    return correct / total

In [104]:
bigram_counts = defaultdict(lambda: defaultdict(int))

for i in range(len(train_data) - 1):
    current_token = train_data[i]
    next_token = train_data[i + 1]

    bigram_counts[current_token][next_token] += 1

In [105]:
def generate_bigram(start_word, max_length=50):
    if start_word not in stoi:
        return f"'{start_word}' not in vocabulary."
    current_token = stoi[start_word]
    generated = [start_word]
    for _ in range(max_length):
        if current_token not in bigram_counts:
            break
        next_tokens = list(bigram_counts[current_token].keys())
        counts = list(bigram_counts[current_token].values())
        next_token = random.choices(next_tokens,weights=counts,k=1)[0]
        generated.append(itos[next_token])
        current_token = next_token
    return " ".join(generated)

In [106]:
def prob_bigram(context, target):
    if context not in bigram_counts:
        return 1e-10
    total = sum(bigram_counts[context].values())
    count = bigram_counts[context].get(target, 0)
    if total == 0:
        return 1e-10
    return count/total

In [107]:
def pred_bigram(context):
    if context not in bigram_counts:
        return None
    next_words = bigram_counts[context]
    return max(next_words.items(),key=lambda x: x[1])[0]

In [108]:
loss = val_loss_bigram(val_data, prob_bigram)
acc = tok_acc_bigram(val_data, pred_bigram)

print(f"Validation Loss: {loss:.4f}")
print(f"Token Accuracy: {acc:.2%}")

Validation Loss: 16.1933
Token Accuracy: 6.49%


In [109]:
def train(model,train_loader,val_loader,epochs=10):
    
    criterion = nn.CrossEntropyLoss() 
    optimizer = tc.optim.Adam(model.parameters(),lr=1e-3)
    
    train_losses = []
    val_loss_bigrames = []
    for epoch in range(epochs):
        
        #Training
        model.train()
        running_loss = 0.0
        train_bar = tqdm(train_loader,desc=f"Epoch {epoch+1}/{epochs} [Train]",leave=False)
        
        for x_batch, y_batch in train_bar:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            optimizer.zero_grad()
            logits = model(x_batch)
            loss = criterion(logits.reshape(-1, logits.size(-1)),y_batch.reshape(-1))
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        avg_train_loss = running_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # Validation
        model.eval()
        running_val_loss_bigram = 0.0
        val_bar = tqdm(val_loader,desc=f"Epoch {epoch+1}/{epochs} [Val]",leave=False)
        
        with tc.no_grad():
            for x_batch, y_batch in val_bar:
                x_batch = x_batch.to(device)
                y_batch = y_batch.to(device)
                logits = model(x_batch)
                loss = criterion(logits.reshape(-1, logits.size(-1)),y_batch.reshape(-1))
                running_val_loss_bigram += loss.item()
        avg_val_loss_bigram = running_val_loss_bigram / len(val_loader)
        val_loss_bigrames.append(avg_val_loss_bigram)
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss_bigram:.4f}")

    return train_losses, val_loss_bigrames

In [110]:
def tok_acc(model, loader):
    model.eval()
    correct = 0
    total = 0
    with tc.no_grad():
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(x_batch)

            preds = logits.argmax(dim=-1)

            correct += (preds == y_batch).sum().item()
            total += y_batch.numel()

    return 100 * correct / total

In [111]:
SEQ_LEN = 20

X_train = []
Y_train = []

for i in range(len(train_data) - SEQ_LEN):
    X_train.append(train_data[i:i+SEQ_LEN])
    Y_train.append(train_data[i+1:i+SEQ_LEN+1])

X_val = []
Y_val = []

for i in range(len(val_data) - SEQ_LEN):
    X_val.append(val_data[i:i+SEQ_LEN])
    Y_val.append(val_data[i+1:i+SEQ_LEN+1])

In [112]:
BATCH_SIZE = 32

X_train_T = tc.tensor(X_train, dtype=tc.long)
Y_train_T = tc.tensor(Y_train, dtype=tc.long)
X_val_T = tc.tensor(X_val, dtype=tc.long)
Y_val_T = tc.tensor(Y_val, dtype=tc.long)

train_dataset = TensorDataset(X_train_T, Y_train_T)
val_dataset = TensorDataset(X_val_T, Y_val_T)

train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)
val_loader = DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False)

In [116]:
class LSTM(nn.Module):
    def __init__(self,vocab_size,embed_dim=64,hidden_dim=128,num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,embed_dim)
        self.lstm = nn.LSTM(input_size=embed_dim,hidden_size=hidden_dim,num_layers=num_layers,dropout=0.3,batch_first=True)
        self.fc = nn.Linear(hidden_dim,vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        output, _ = self.lstm(x)
        logits = self.fc(output)
        return logits

In [117]:
model = LSTM(vocab_size=vocab_size).to(device)
train_losses, val_losses = train(model,train_loader,val_loader,epochs=10)

Epoch 1/10 | Train Loss: 6.9401 | Val Loss: 8.0773


Epoch 2/10 | Train Loss: 5.7440 | Val Loss: 8.5253


Epoch 3/10 | Train Loss: 4.7957 | Val Loss: 9.0382


Epoch 4/10 | Train Loss: 4.1104 | Val Loss: 9.6429


Epoch 5/10 | Train Loss: 3.6463 | Val Loss: 10.1817


Epoch 6/10 | Train Loss: 3.3075 | Val Loss: 10.6726


Epoch 7/10 | Train Loss: 3.0414 | Val Loss: 11.1010


Epoch 8/10 | Train Loss: 2.8211 | Val Loss: 11.4695


Epoch 9/10 | Train Loss: 2.6375 | Val Loss: 11.8055


Epoch 10/10 | Train Loss: 2.4774 | Val Loss: 12.0942


In [118]:
acc = tok_acc(model, val_loader)
print(f"Validation Accuracy: {acc:.2f}%")

Validation Accuracy: 8.83%
